# TTS Benchmark: VieNeu-TTS vs MMS-TTS vs F5-TTS
So sánh RTF, TTFA (Time to First Audio) và WER intelligibility trên 40 câu tiếng Việt dinh dưỡng.

**Models:**
- VieNeu-TTS 0.3B q4 (baseline, streaming, GPU)
- MMS-TTS `facebook/mms-tts-vie` (VITS, GPU)
- F5-TTS `hynt/F5-TTS-Vietnamese-ViVoice` (Flow Matching, 1000h, GPU)

**Metrics:**
- **RTF** = inference_time / audio_duration (< 1.0 = real-time)
- **TTFA** = thời gian đến byte âm thanh đầu tiên (ms)
- **WER** = PhoWhisper-medium transcribe audio tổng hợp → so với input text

**Upload lên Drive trước khi chạy:**
```
callbot_tts/
```
(Thư mục này chỉ chứa kết quả, không cần upload model — tất cả download từ HuggingFace)

In [ ]:
%%capture
# llama-cpp-python với CUDA (cho VieNeu-TTS backbone)
# Nếu dùng Colab CUDA 12.4:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 -q
# Fallback nếu dòng trên lỗi (compile từ source, ~10 phút):
# !CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir -q

!pip install git+https://github.com/pnnbao97/VieNeu-TTS.git -q
!pip install transformers accelerate -q
!pip install f5-tts -q
!pip install faster-whisper jiwer scipy soundfile huggingface_hub -q

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
DRIVE_BASE  = "/content/drive/MyDrive/callbot_tts"
RESULTS_DIR = "/content/tts_results"
TMP_DIR     = "/content/tts_tmp"   # temp WAV files cho WER eval

import os
from pathlib import Path
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(DRIVE_BASE, exist_ok=True)
TMP_DIR = Path(TMP_DIR)
print("Paths OK")

In [ ]:
# ── 40 câu test tiếng Việt (dinh dưỡng, 3 nhóm độ dài) ────────────
TEST_SENTENCES = [
    # Short — ≤10 từ (15 câu)
    "Cá hồi rất giàu axit béo omega-3.",
    "Uống đủ nước mỗi ngày rất quan trọng.",
    "Rau xanh cung cấp nhiều vitamin thiết yếu.",
    "Đường tinh luyện có hại cho sức khỏe.",
    "Canxi giúp xương và răng chắc khỏe.",
    "Protein cần thiết cho cơ bắp phát triển.",
    "Hạt chia giàu chất xơ và omega-3.",
    "Muối nhiều dẫn đến tăng huyết áp.",
    "Trứng chứa nhiều protein và vitamin D.",
    "Quả bơ là nguồn chất béo lành mạnh.",
    "Kẽm tăng cường hệ miễn dịch hiệu quả.",
    "Sắt có nhiều trong thịt đỏ và rau bina.",
    "Chất xơ hỗ trợ tiêu hóa tốt hơn.",
    "Mỡ nội tạng nguy hiểm cho tim mạch.",
    "Magie cần thiết cho chức năng thần kinh.",

    # Medium — 11–20 từ (15 câu)
    "Người trưởng thành cần khoảng hai nghìn kilocalorie mỗi ngày để duy trì cân nặng.",
    "Vitamin C có trong cam, chanh và ổi giúp tăng sức đề kháng cho cơ thể.",
    "Protein động vật từ thịt và cá cung cấp đầy đủ các axit amin thiết yếu.",
    "Chế độ ăn ít muối giúp giảm nguy cơ tăng huyết áp và bảo vệ tim mạch.",
    "Sắt có trong thịt đỏ và các loại đậu giúp ngăn ngừa thiếu máu hiệu quả.",
    "Omega-3 trong cá thu và cá hồi giúp giảm cholesterol xấu trong máu.",
    "Trẻ em cần bổ sung canxi và vitamin D để xương phát triển bình thường.",
    "Chất béo bão hòa trong thịt mỡ và bơ động vật nên được hạn chế tiêu thụ.",
    "Rau lá xanh đậm như cải bó xôi và súp lơ xanh rất giàu folate và vitamin K.",
    "Đường tiêu thụ quá mức có thể dẫn đến béo phì và đái tháo đường type hai.",
    "Probiotics trong sữa chua giúp cân bằng hệ vi sinh đường ruột và tăng cường miễn dịch.",
    "Ăn sáng đầy đủ giúp duy trì năng lượng và khả năng tập trung trong suốt buổi sáng.",
    "Yến mạch có chỉ số glycemic thấp giúp kiểm soát lượng đường trong máu ổn định hơn.",
    "Vitamin B12 chủ yếu có trong thực phẩm động vật, người ăn chay cần bổ sung thêm.",
    "Kali trong chuối và khoai lang giúp cân bằng điện giải và ổn định huyết áp trong cơ thể.",

    # Long — >20 từ (10 câu)
    "Theo khuyến nghị của Tổ chức Y tế Thế giới, người lớn nên ăn ít nhất năm phần rau và trái cây mỗi ngày để bảo vệ sức khỏe và phòng ngừa các bệnh mạn tính.",
    "Chất béo không bão hòa đơn và đa có trong dầu ô liu, quả bơ và các loại hạt có lợi cho sức khỏe tim mạch, giúp giảm cholesterol xấu và tăng cholesterol tốt trong máu.",
    "Người bị đái tháo đường type hai nên ưu tiên thực phẩm có chỉ số đường huyết thấp như rau củ, ngũ cốc nguyên hạt và các loại đậu để kiểm soát lượng đường trong máu.",
    "Trong giai đoạn mang thai, phụ nữ cần bổ sung thêm axit folic, sắt và canxi để hỗ trợ sự phát triển của thai nhi và duy trì sức khỏe toàn diện của người mẹ.",
    "Chế độ ăn Địa Trung Hải với nhiều rau củ, cá, dầu ô liu và ít thịt đỏ được các chuyên gia dinh dưỡng đánh giá là một trong những chế độ ăn lành mạnh nhất thế giới.",
    "Thiếu sắt là nguyên nhân phổ biến nhất gây thiếu máu ở phụ nữ trong độ tuổi sinh sản và trẻ em, biểu hiện qua triệu chứng mệt mỏi, hoa mắt và da xanh xao.",
    "Natri trong muối ăn và thực phẩm chế biến sẵn như mì gói, đồ hộp và thức ăn nhanh là yếu tố nguy cơ chính gây tăng huyết áp và các bệnh tim mạch nguy hiểm.",
    "Ăn đa dạng thực phẩm từ nhiều nhóm khác nhau bao gồm ngũ cốc, rau củ, trái cây, đạm và sữa giúp đảm bảo cung cấp đầy đủ các chất dinh dưỡng cần thiết cho cơ thể.",
    "Vitamin D được tổng hợp chủ yếu qua da khi tiếp xúc với ánh sáng mặt trời, và cũng có trong cá béo, lòng đỏ trứng và sữa tăng cường để hỗ trợ hấp thu canxi hiệu quả.",
    "Chế độ ăn nhiều rau xanh, trái cây và ít thịt đỏ chế biến không chỉ giúp duy trì cân nặng hợp lý mà còn giảm đáng kể nguy cơ mắc các bệnh ung thư đại trực tràng.",
]

print(f"Tổng số câu: {len(TEST_SENTENCES)}")
print(f"Short (≤10 từ): {sum(1 for s in TEST_SENTENCES if len(s.split()) <= 10)}")
print(f"Medium (11-20 từ): {sum(1 for s in TEST_SENTENCES if 10 < len(s.split()) <= 20)}")
print(f"Long (>20 từ): {sum(1 for s in TEST_SENTENCES if len(s.split()) > 20)}")

In [ ]:
# ── Helper functions ───────────────────────────────────────────────
import re
import time
import numpy as np
import scipy.io.wavfile as wavfile
from jiwer import wer as jiwer_wer

def normalize_vi(text: str) -> str:
    """Lowercase + xóa dấu câu, giữ ký tự tiếng Việt."""
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def compute_wer(ref: str, hyp: str) -> float:
    return jiwer_wer(normalize_vi(ref), normalize_vi(hyp))

def save_wav_f32(audio_np: np.ndarray, sr: int, path):
    """Lưu float32 array (range -1..1) thành PCM int16 WAV."""
    audio_i16 = (np.clip(audio_np, -1.0, 1.0) * 32767).astype(np.int16)
    wavfile.write(str(path), sr, audio_i16)

def audio_duration(audio_np: np.ndarray, sr: int) -> float:
    return len(audio_np) / sr

def get_length_bucket(text: str) -> str:
    n = len(text.split())
    if n <= 10:
        return "short (≤10 từ)"
    elif n <= 20:
        return "medium (11-20 từ)"
    return "long (>20 từ)"

def transcribe_for_wer(whisper_model, wav_path: str) -> str:
    segs, _ = whisper_model.transcribe(str(wav_path), language="vi", beam_size=5)
    return " ".join(s.text for s in segs).strip()

def run_tts_benchmark(model_name, synthesize_fn, sentences, whisper_model, tmp_dir):
    """
    synthesize_fn(text) -> (audio_np: float32, sr: int, ttfa_ms: float|None, inf_ms: float)
      ttfa_ms = None nếu model không hỗ trợ streaming (sẽ dùng inf_ms thay thế)
    """
    print(f"\nBenchmarking {model_name} ({len(sentences)} câu)...")
    results = []
    for i, text in enumerate(sentences):
        try:
            audio_np, sr, ttfa_ms, inf_ms = synthesize_fn(text)
            dur_s = audio_duration(audio_np, sr)
            rtf = inf_ms / 1000 / max(dur_s, 0.001)

            # Lưu WAV tạm để PhoWhisper transcribe
            wav_path = tmp_dir / f"{model_name.replace('/', '_')}_{i:03d}.wav"
            save_wav_f32(audio_np, sr, wav_path)
            hyp = transcribe_for_wer(whisper_model, wav_path)
            wer_val = compute_wer(text, hyp)

            results.append({
                "id": i,
                "text": text,
                "hyp": hyp,
                "wer": wer_val,
                "duration_s": dur_s,
                "inf_ms": inf_ms,
                "ttfa_ms": ttfa_ms if ttfa_ms is not None else inf_ms,
                "rtf": rtf,
                "length_bucket": get_length_bucket(text),
                "streaming": ttfa_ms is not None,
            })

            if (i + 1) % 10 == 0 or i == 0:
                print(f"  {i+1:2d}/{len(sentences)} | wer={wer_val:.1%} rtf={rtf:.3f} ttfa={results[-1]['ttfa_ms']:.0f}ms")

        except Exception as e:
            print(f"  [{i}] ERROR: {e}")
            results.append({"id": i, "text": text, "error": str(e)})

    return [r for r in results if "error" not in r]

def compute_metrics(results, model_name):
    wers  = [r["wer"]  for r in results]
    rtfs  = [r["rtf"]  for r in results]
    ttfas = [r["ttfa_ms"] for r in results]
    infs  = [r["inf_ms"]  for r in results]
    durs  = [r["duration_s"] for r in results]

    wer_by_len = {}
    for bucket in ["short (≤10 từ)", "medium (11-20 từ)", "long (>20 từ)"]:
        sub = [r for r in results if r["length_bucket"] == bucket]
        if sub:
            wer_by_len[bucket] = np.mean([r["wer"] for r in sub])

    return {
        "model": model_name,
        "n": len(results),
        "wer_mean": np.mean(wers),
        "rtf_mean": np.mean(rtfs),
        "rtf_p50": np.percentile(rtfs, 50),
        "rtf_p90": np.percentile(rtfs, 90),
        "ttfa_mean_ms": np.mean(ttfas),
        "ttfa_p90_ms": np.percentile(ttfas, 90),
        "inf_mean_ms": np.mean(infs),
        "inf_p90_ms": np.percentile(infs, 90),
        "wer_by_length": wer_by_len,
        "rtf_values": rtfs,
        "ttfa_values": ttfas,
        "dur_values": durs,
        "inf_values": infs,
    }

all_metrics = {}
all_results = {}
print("Helpers OK")

In [ ]:
# ── Load PhoWhisper-medium (dùng cho toàn bộ WER eval) ────────────
# Giữ model này trong suốt notebook — chỉ load 1 lần
from faster_whisper import WhisperModel
print("Loading PhoWhisper-medium...")
phowhisper = WhisperModel("vinai/PhoWhisper-medium", device="cuda", compute_type="float16")
print("PhoWhisper-medium ready")

## Model 1: VieNeu-TTS 0.3B (baseline, streaming)

In [ ]:
# VieNeu-TTS — backbone chạy trên GPU (llama.cpp CUDA), codec trên CUDA
from vieneu import Vieneu

print("Loading VieNeu-TTS 0.3B...")
tts_vieneu = Vieneu(
    backbone_repo="pnnbao-ump/VieNeu-TTS-0.3B-q4-gguf",
    backbone_device="gpu",    # llama.cpp dùng 'gpu'
    codec_repo="neuphonic/distill-neucodec",
    codec_device="cuda",      # PyTorch dùng 'cuda'
)
vieneu_voice = tts_vieneu.get_preset_voice()
print("VieNeu-TTS ready")

def synthesize_vieneu(text):
    t0 = time.perf_counter()
    ttfa_ms = None
    chunks = []
    for chunk in tts_vieneu.infer_stream(
        text=text, voice=vieneu_voice,
        max_chars=256, temperature=1.0, top_k=50,
    ):
        if ttfa_ms is None:
            ttfa_ms = (time.perf_counter() - t0) * 1000
        chunks.append(chunk)
    inf_ms = (time.perf_counter() - t0) * 1000
    audio = np.concatenate(chunks).astype(np.float32) if chunks else np.zeros(1, dtype=np.float32)
    return audio, 24000, ttfa_ms, inf_ms

# Warmup
_ = synthesize_vieneu("Xin chào.")

# Synthesis reference audio (dùng cho F5-TTS sau)
REF_TEXT_FOR_F5 = "Tôi là chuyên gia tư vấn dinh dưỡng, rất vui được hỗ trợ bạn tìm hiểu về chế độ ăn lành mạnh."
ref_audio, ref_sr, _, _ = synthesize_vieneu(REF_TEXT_FOR_F5)
REF_WAV = "/content/ref_voice.wav"
save_wav_f32(ref_audio, ref_sr, REF_WAV)
print(f"Saved reference audio: {REF_WAV} ({audio_duration(ref_audio, ref_sr):.1f}s)")

# Benchmark
results_vieneu = run_tts_benchmark("VieNeu-TTS", synthesize_vieneu, TEST_SENTENCES, phowhisper, TMP_DIR)
all_results["VieNeu-TTS"] = results_vieneu
all_metrics["VieNeu-TTS"] = compute_metrics(results_vieneu, "VieNeu-TTS")
print(f"\nVieNeu-TTS: WER={all_metrics['VieNeu-TTS']['wer_mean']:.1%}  RTF={all_metrics['VieNeu-TTS']['rtf_mean']:.3f}  TTFA={all_metrics['VieNeu-TTS']['ttfa_mean_ms']:.0f}ms")

In [ ]:
# Unload VieNeu-TTS
del tts_vieneu, vieneu_voice
import gc; gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1e6:.0f}MB allocated")

## Model 2: MMS-TTS (facebook/mms-tts-vie)

In [ ]:
from transformers import VitsModel, AutoTokenizer

print("Loading MMS-TTS (facebook/mms-tts-vie)...")
mms_model = VitsModel.from_pretrained("facebook/mms-tts-vie").to("cuda")
mms_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-vie")
mms_sr = mms_model.config.sampling_rate
print(f"MMS-TTS ready | sample_rate={mms_sr}Hz")

def synthesize_mms(text):
    inputs = mms_tokenizer(text, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    t0 = time.perf_counter()
    with torch.no_grad():
        output = mms_model(**inputs).waveform
    inf_ms = (time.perf_counter() - t0) * 1000
    audio = output.squeeze().cpu().numpy().astype(np.float32)
    # Normalize nếu cần
    peak = np.abs(audio).max()
    if peak > 1.0:
        audio = audio / peak
    return audio, mms_sr, None, inf_ms  # không streaming → ttfa = inf_ms

# Warmup
_ = synthesize_mms("Xin chào.")

results_mms = run_tts_benchmark("MMS-TTS", synthesize_mms, TEST_SENTENCES, phowhisper, TMP_DIR)
all_results["MMS-TTS"] = results_mms
all_metrics["MMS-TTS"] = compute_metrics(results_mms, "MMS-TTS")
print(f"\nMMS-TTS: WER={all_metrics['MMS-TTS']['wer_mean']:.1%}  RTF={all_metrics['MMS-TTS']['rtf_mean']:.3f}  TTFA={all_metrics['MMS-TTS']['ttfa_mean_ms']:.0f}ms")

In [ ]:
# Unload MMS-TTS
del mms_model, mms_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1e6:.0f}MB allocated")

## Model 3: F5-TTS ViVoice (hynt/F5-TTS-Vietnamese-ViVoice)

Model 1000h tiếng Việt, fine-tune từ F5-TTS_v1_Base.  
**Zero-shot voice cloning** — dùng reference audio từ VieNeu-TTS (đã tổng hợp ở Cell 9).

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

REPO_F5 = "hynt/F5-TTS-Vietnamese-ViVoice"
print(f"Listing files in {REPO_F5}...")
repo_files = list(list_repo_files(REPO_F5))
print("Files:", repo_files)

# Download checkpoint
pt_files = [f for f in repo_files if f.endswith(".pt")]
if not pt_files:
    raise FileNotFoundError(f"Không tìm thấy .pt file trong {REPO_F5}")
ckpt_path = hf_hub_download(REPO_F5, pt_files[0])
print(f"Checkpoint: {ckpt_path}")

# Vocab file (nếu có)
vocab_files = [f for f in repo_files if "vocab" in f.lower() and not f.endswith(".py")]
vocab_path = hf_hub_download(REPO_F5, vocab_files[0]) if vocab_files else ""
print(f"Vocab: {vocab_path or '(dùng default F5-TTS)'}")

In [ ]:
from f5_tts.api import F5TTS

print("Loading F5-TTS ViVoice...")
f5_model = F5TTS(
    model_type="F5TTS_v1_Base",
    ckpt_file=str(ckpt_path),
    vocab_file=str(vocab_path) if vocab_path else "",
)
print("F5-TTS ViVoice ready")

# Reference audio từ VieNeu-TTS (Cell 9)
print(f"Reference audio: {REF_WAV}")
print(f"Reference text : {REF_TEXT_FOR_F5}")

_f5_tmp = TMP_DIR / "_f5_inference_tmp.wav"

def synthesize_f5(text):
    t0 = time.perf_counter()
    wav, sr, _ = f5_model.infer(
        ref_file=REF_WAV,
        ref_text=REF_TEXT_FOR_F5,
        gen_text=text,
        file_wave=str(_f5_tmp),
        remove_silence=True,
        show_info=lambda *a, **k: None,  # suppress progress output
    )
    inf_ms = (time.perf_counter() - t0) * 1000
    audio = np.array(wav, dtype=np.float32)
    if audio.dtype != np.float32 or audio.max() > 1.0:
        audio = audio.astype(np.float32) / max(np.abs(audio).max(), 1.0)
    return audio, int(sr), None, inf_ms  # không streaming

# Warmup
_ = synthesize_f5("Xin chào.")

results_f5 = run_tts_benchmark("F5-TTS-ViVoice", synthesize_f5, TEST_SENTENCES, phowhisper, TMP_DIR)
all_results["F5-TTS-ViVoice"] = results_f5
all_metrics["F5-TTS-ViVoice"] = compute_metrics(results_f5, "F5-TTS-ViVoice")
print(f"\nF5-TTS: WER={all_metrics['F5-TTS-ViVoice']['wer_mean']:.1%}  RTF={all_metrics['F5-TTS-ViVoice']['rtf_mean']:.3f}  TTFA={all_metrics['F5-TTS-ViVoice']['ttfa_mean_ms']:.0f}ms")

In [ ]:
# Unload F5-TTS
del f5_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1e6:.0f}MB allocated")

## Kết quả

In [ ]:
# ── Summary table ──────────────────────────────────────────────────
print("=" * 80)
print("KẾT QUẢ BENCHMARK TTS")
print("=" * 80)
print(f"{'Model':<20} {'WER':>7} {'RTF_mean':>9} {'RTF_p90':>8} {'TTFA_mean':>10} {'TTFA_p90':>9} {'Inf_mean':>9}")
print("-" * 80)
for name, m in all_metrics.items():
    streaming = any(r["streaming"] for r in all_results[name])
    stream_flag = "(stream)" if streaming else ""
    print(f"{name+stream_flag:<20} {m['wer_mean']*100:>6.1f}% "
          f"{m['rtf_mean']:>9.3f} {m['rtf_p90']:>8.3f} "
          f"{m['ttfa_mean_ms']:>9.0f}ms {m['ttfa_p90_ms']:>8.0f}ms "
          f"{m['inf_mean_ms']:>8.0f}ms")

print("\n── WER theo độ dài câu ──────────────────────────────────────")
buckets = ["short (≤10 từ)", "medium (11-20 từ)", "long (>20 từ)"]
print(f"{'Model':<20}", end="")
for b in buckets:
    print(f" {b:>17}", end="")
print()
for name, m in all_metrics.items():
    print(f"{name:<20}", end="")
    for b in buckets:
        v = m["wer_by_length"].get(b)
        print(f" {v*100:>16.1f}%" if v is not None else f" {'N/A':>17}", end="")
    print()
print("=" * 80)

In [ ]:
# ── Charts ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

models  = list(all_metrics.keys())
colors  = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759"]
buckets = ["short (≤10 từ)", "medium (11-20 từ)", "long (>20 từ)"]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("TTS Benchmark: VieNeu-TTS vs MMS-TTS vs F5-TTS ViVoice",
             fontsize=13, fontweight="bold")

# 1. WER bar + by length bucket (grouped)
ax = axes[0, 0]
x = np.arange(len(buckets))
width = 0.25
for i, name in enumerate(models):
    vals = [all_metrics[name]["wer_by_length"].get(b, 0) * 100 for b in buckets]
    bars = ax.bar(x + i * width, vals, width, label=name, color=colors[i], edgecolor="white")
ax.set_title("WER theo độ dài câu (thấp hơn tốt hơn)")
ax.set_ylabel("WER (%)")
ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels([b.replace(" (", "\n(") for b in buckets], fontsize=8)
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# 2. RTF — mean + p90 bar chart
ax = axes[0, 1]
x2 = np.arange(len(models))
w = 0.35
bars_mean = ax.bar(x2 - w/2, [all_metrics[m]["rtf_mean"] for m in models], w,
                   label="mean", color=[colors[i] for i in range(len(models))], edgecolor="white", alpha=0.85)
bars_p90  = ax.bar(x2 + w/2, [all_metrics[m]["rtf_p90"]  for m in models], w,
                   label="p90",  color=[colors[i] for i in range(len(models))], edgecolor="white", alpha=0.45)
ax.axhline(1.0, color="red", linestyle="--", linewidth=1, alpha=0.6, label="RTF=1 (real-time limit)")
ax.set_title("RTF (Real-Time Factor) — thấp hơn nhanh hơn")
ax.set_ylabel("RTF")
ax.set_xticks(x2)
ax.set_xticklabels(models, rotation=10, ha="right", fontsize=8)
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
ax.bar_label(bars_mean, fmt="%.3f", padding=2, fontsize=7)
ax.bar_label(bars_p90,  fmt="%.3f", padding=2, fontsize=7)

# 3. TTFA — mean + p90 bar chart
ax = axes[1, 0]
bars_tmean = ax.bar(x2 - w/2, [all_metrics[m]["ttfa_mean_ms"] for m in models], w,
                    label="mean", color=[colors[i] for i in range(len(models))], edgecolor="white", alpha=0.85)
bars_tp90  = ax.bar(x2 + w/2, [all_metrics[m]["ttfa_p90_ms"]  for m in models], w,
                    label="p90",  color=[colors[i] for i in range(len(models))], edgecolor="white", alpha=0.45)
ax.axhline(300, color="green", linestyle="--", linewidth=1, alpha=0.6, label="300ms target")
ax.set_title("TTFA (Time to First Audio) — thấp hơn tốt hơn\n* MMS/F5 không streaming → TTFA = toàn bộ synthesis")
ax.set_ylabel("TTFA (ms)")
ax.set_xticks(x2)
ax.set_xticklabels(models, rotation=10, ha="right", fontsize=8)
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
ax.bar_label(bars_tmean, fmt="%.0f", padding=2, fontsize=7)

# 4. RTF CDF
ax = axes[1, 1]
for i, (name, m) in enumerate(all_metrics.items()):
    sorted_rtf = sorted(m["rtf_values"])
    cdf = np.arange(1, len(sorted_rtf) + 1) / len(sorted_rtf)
    ax.plot(sorted_rtf, cdf, label=name, color=colors[i], linewidth=2)
ax.axvline(1.0, color="red", linestyle="--", linewidth=1, alpha=0.6, label="RTF=1")
ax.set_title("CDF của RTF — đường càng trái càng nhanh")
ax.set_xlabel("RTF")
ax.set_ylabel("Tỷ lệ tích lũy")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()

from datetime import datetime
tag = datetime.now().strftime("%Y%m%d_%H%M%S")
chart_path = f"{RESULTS_DIR}/tts_charts_{tag}.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart_path}")

In [ ]:
# ── Save CSV + JSON ─────────────────────────────────────────────────
import csv, json

# Summary CSV
csv_path = f"{RESULTS_DIR}/tts_summary_{tag}.csv"
fieldnames = ["model", "n", "wer_mean", "rtf_mean", "rtf_p50", "rtf_p90",
              "ttfa_mean_ms", "ttfa_p90_ms", "inf_mean_ms", "inf_p90_ms"]
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for name, m in all_metrics.items():
        w.writerow({k: m[k] for k in fieldnames})

# Per-sentence JSONL
jsonl_path = f"{RESULTS_DIR}/tts_results_{tag}.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for model_name, results in all_results.items():
        for r in results:
            row = {"model": model_name, **{k: v for k, v in r.items() if k != "audio"}}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"CSV    : {csv_path}")
print(f"JSONL  : {jsonl_path}")
print(f"Charts : {chart_path}")

In [ ]:
# ── Copy về Drive ───────────────────────────────────────────────────
import shutil
dst = Path(DRIVE_BASE) / "results"
dst.mkdir(parents=True, exist_ok=True)
for f in Path(RESULTS_DIR).glob(f"*{tag}*"):
    shutil.copy(f, dst / f.name)
print(f"Đã copy kết quả về: {dst}")